# U-Net

Ronneberger, Fischer, Brox, *U-Net: Convolutional Networks for Biomedical Image Segmentation*, MICCAI 2015 ([arXiv:1505.04597](https://arxiv.org/abs/1505.04597)).

Every other model in this repo reduces an image to one label; U-Net repurposes the same conv/pool vocabulary for *dense*, per-pixel prediction, via a contracting encoder + expanding decoder with skip connections carrying fine spatial detail across. This implementation uses `padding=1` convolutions (so no cropping is needed at the skip connections, unlike the paper's unpadded convs) and a reduced channel count for CPU-friendly training time -- see `model.py` and `README.md` for details.

This notebook trains a `UNet` for binary foreground/background segmentation on real Oxford-IIIT Pet photos.

In [ ]:
import sys
sys.path.insert(0, '../..')
sys.path.insert(0, '.')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from cnn_playground.data import load_oxford_pet_segmentation
from cnn_playground.device import resolve_device
from cnn_playground.utils.seed import set_seed
from model import UNet

set_seed(0)
# device options: 'auto' (default, picks cuda/mps if available), 'cpu', 'cuda', 'mps'
device = resolve_device('auto')
print('device:', device)

In [ ]:
train_ds = load_oxford_pet_segmentation(train=True)
test_ds = load_oxford_pet_segmentation(train=False)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
print(len(train_ds), 'train images,', len(test_ds), 'test images')
img, mask = train_ds[0]
print('image', img.shape, 'mask', mask.shape, 'mask values', mask.unique())

In [ ]:
def iou(pred_mask, true_mask):
    intersection = (pred_mask * true_mask).sum(dim=(1, 2, 3))
    union = ((pred_mask + true_mask) > 0).float().sum(dim=(1, 2, 3))
    return (intersection / union.clamp_min(1e-8)).mean().item()

model = UNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

history = {'train_loss': [], 'test_iou': []}
epochs = 15
for epoch in range(epochs):
    model.train()
    last_loss = None
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        opt.zero_grad()
        logits = model(imgs)
        loss = loss_fn(logits, masks)
        loss.backward()
        opt.step()
        last_loss = loss.item()

    model.eval()
    ious = []
    with torch.no_grad():
        for imgs, masks in test_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            pred = (torch.sigmoid(model(imgs)) > 0.5).float()
            ious.append(iou(pred, masks))
    history['train_loss'].append(last_loss)
    history['test_iou'].append(sum(ious) / len(ious))

print(f"final test IoU: {history['test_iou'][-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history['train_loss']); axes[0].set_title('train loss'); axes[0].set_xlabel('epoch')
axes[1].plot(history['test_iou']); axes[1].set_title('test IoU'); axes[1].set_xlabel('epoch')
fig.tight_layout()
plt.show()

In [ ]:
model.eval()
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
with torch.no_grad():
    for i in range(3):
        img, mask = test_ds[i]
        pred = torch.sigmoid(model(img.unsqueeze(0).to(device)))[0, 0].cpu()
        axes[i, 0].imshow(img.permute(1, 2, 0)); axes[i, 0].set_title('image'); axes[i, 0].axis('off')
        axes[i, 1].imshow(mask[0], cmap='gray'); axes[i, 1].set_title('real mask'); axes[i, 1].axis('off')
        axes[i, 2].imshow(pred > 0.5, cmap='gray'); axes[i, 2].set_title('predicted mask'); axes[i, 2].axis('off')
fig.tight_layout()
plt.show()